# 22 · Estimate at posting time with embeddings and context (E52)

E43 estimated views before any reaction exists from followers, text stats, VADER, TF-IDF SVD, AI labels, calendar, horizon and brand size. Here the local `multilingual-e5-small` embeddings (PCA 16) and the context from other posts of the same author, hashtags and brand (E49, without the post's own likes) are added. Same folds and CatBoost settings as E43.

Caveat: context uses likes of other posts in the whole data, also later ones, so it is an upper bound for real posting time.

In [1]:
import sys, json, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw" / "reach_samples_students.csv").exists())
sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.width", 220, "display.max_columns", 30)
import history as H, posting_time_text as pt

In [2]:
groups, meta, tfidf = pt.data(ROOT, ROOT / "data" / "processed" / "embeddings_e5_cache.pkl")
print(len(tfidf), "TF-IDF columns in the E43 set;", {k: v.shape[1] for k, v in groups.items()})

30 TF-IDF columns in the E43 set; {'followers': 2, 'posting time': 56, 'reactions': 6, 'posting time, no TF-IDF': 26, 'embeddings': 16, 'context': 10}


In [3]:
P = pt.run(groups, meta); S = H.summarise(P)
S.pivot(index="model", columns="platform", values="MAE")[pt.PLATS].loc[list(pt.SETS)].round(3)

platform,tiktok,youtube,twitter
model,,,
E43 posting-time columns,0.575,0.684,0.550
+ embeddings,0.570,0.678,0.544
TF-IDF replaced by embeddings,0.566,0.689,0.542
+ context,0.570,0.664,0.542
+ embeddings + context,0.567,0.667,0.540


In [4]:
G = pd.DataFrame([{"platform": p, "model": k, **dict(zip(["gain", "CI low", "CI high"], H.gain_ci(P, p, pt.BASE, k)))} for p in pt.PLATS for k in list(pt.SETS)[1:]]); G.round(3)

,platform,model,gain,CI low,CI high
0,tiktok,+ embeddings,0.005,0.001,0.009
1,tiktok,TF-IDF replaced by embeddings,0.009,0.003,0.016
2,tiktok,+ context,0.004,-0.000,0.009
3,tiktok,+ embeddings + context,0.008,0.003,0.013
4,youtube,+ embeddings,0.006,-0.000,0.012
5,youtube,TF-IDF replaced by embeddings,-0.005,-0.016,0.006
6,youtube,+ context,0.020,0.011,0.027
7,youtube,+ embeddings + context,0.017,0.008,0.027
8,twitter,+ embeddings,0.006,0.000,0.012
9,twitter,TF-IDF replaced by embeddings,0.008,-0.001,0.018


## Takeaways
See `docs/en/experiments.md` (E52) and `agent/vault/E52.md`. Numbers: `data/processed/posting_time_text_numbers.json`.

In [5]:
json.dump({"models": S.to_dict("records"), "gains": G.to_dict("records")}, open(ROOT / "data" / "processed" / "posting_time_text_numbers.json", "w"), indent=1, default=float)